# DATA SCIENCE METHODOLOGY – 10 BƯỚC
## Phân tích các yếu tố ảnh hưởng đến kết quả học tập của sinh viên bằng SQL và Python
**Bộ dữ liệu:** `StudentPerformanceFactors.csv` (6.607 bản ghi, 16 cột). **Trọng tâm:** truy xuất, làm sạch, phân tích dữ liệu SQL/Python. 10 bước tương ứng sơ đồ *Data Science Methodology* (FPT University): Business understanding → Analytic approach → Data requirements → Data collection → Data understanding → Data preparation → Modeling → Evaluation → Deployment → Feedback.

**Cách chạy:** đặt notebook cùng thư mục CSV, chọn **Run All**. Chỉ dùng thư viện `pandas`, `numpy` và `sqlite3` (sqlite3 có sẵn trong Python).

## Bước 1 — Business Understanding: Đặt mục tiêu
**Vấn đề cần giải quyết (*What is the problem that you are trying to solve?*):** Nhà trường muốn biết những yếu tố học tập và hoàn cảnh nào đi cùng với điểm thi (`Exam_Score`) cao hoặc thấp, để ưu tiên hỗ trợ đúng nhóm sinh viên.

**Câu hỏi phân tích:** (1) Điểm thi trung bình của sinh viên là bao nhiêu? (2) Các nhóm số giờ tự học/tuần và tỷ lệ chuyên cần khác nhau có điểm khác nhau như thế nào? (3) Kết quả khác nhau ra sao giữa các nhóm động lực học tập, sự tham gia của phụ huynh, học vấn phụ huynh và hoạt động ngoại khóa?

**Đầu ra:** dữ liệu đã làm sạch, truy vấn SQL, bảng thống kê và báo cáo tóm tắt. Chỉ phân tích mối liên hệ; không khẳng định quan hệ nhân quả.

## Bước 2 — Analytic Approach: Hướng tiếp cận
Phân tích **mô tả + so sánh nhóm**. Sử dụng **SQL** để kiểm tra dữ liệu, lọc, JOIN, GROUP BY, CASE WHEN, CTE và window functions. Sử dụng **Python/Pandas** để làm sạch, đối chiếu kết quả SQL và phân tích định lượng đơn giản (tương quan Pearson). Không cần mô hình học máy.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("StudentPerformanceFactors.csv")
assert DATA.exists(), f"Không tìm thấy {DATA}. Đặt CSV cùng thư mục notebook."
conn = sqlite3.connect("student_analysis.db")
print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.50.4


## Bước 3 — Data Requirements: Xác định dữ liệu cần dùng
Biến đầu vào chính: `Hours_Studied` (giờ tự học/tuần), `Attendance` (%), `Sleep_Hours`, `Previous_Scores`, `Motivation_Level`, `Parental_Involvement`, `Parental_Education_Level`, `Extracurricular_Activities`; biến kết quả: `Exam_Score`. Các trường khác (`Access_to_Resources`, `Family_Income`, `Teacher_Quality`, `School_Type`, `Peer_Influence`, `Physical_Activity`, `Gender`) được giữ để hỗ trợ truy vấn. Dữ liệu gốc **không có cột khóa** → tạo `student_id` theo thứ tự dòng (`rowid` trong SQLite = chỉ số dòng + 1 trong Python).

In [2]:
raw = pd.read_csv(DATA)
print("Kích thước:", raw.shape)
print("Các cột:", raw.columns.tolist())
print(raw.head(5).to_string(index=False))
raw.to_sql("raw_students", conn, if_exists="replace", index=False)
print("Đã nạp dữ liệu vào SQLite: raw_students")

Kích thước: (6607, 16)
Các cột: ['Hours_Studied', 'Attendance', 'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours', 'Previous_Scores', 'Motivation_Level', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Physical_Activity', 'Parental_Education_Level', 'Gender', 'Exam_Score']
 Hours_Studied  Attendance Parental_Involvement Access_to_Resources Extracurricular_Activities  Sleep_Hours  Previous_Scores Motivation_Level Family_Income Teacher_Quality School_Type Peer_Influence  Physical_Activity Parental_Education_Level Gender  Exam_Score
            23          84                  Low                High                         No            7               73              Low           Low          Medium      Public       Positive                  3              High School   Male          67
            19          64                  Low              Medium                         No            8               59              L

## Bước 4 — Data Collection: Truy xuất dữ liệu bằng SQL và Python
`pd.read_csv` đọc CSV; `to_sql` nạp vào SQLite; `pd.read_sql_query` lấy kết quả truy vấn từ SQLite ra DataFrame Python.

- **Truy vấn 4.1:** 10 sinh viên điểm cao nhất trong nhóm chuyên cần ≥ 90% (`WHERE`, `ORDER BY`, `LIMIT`; `rowid` dùng làm `student_id`).
- **Truy vấn 4.2:** quy mô và điểm trung bình theo loại trường và giới tính (`GROUP BY`) — đối chiếu bằng Pandas `groupby`.

In [3]:
sql_4 = """
SELECT rowid AS student_id, Gender, Hours_Studied, Attendance,
       Motivation_Level, Parental_Education_Level, Exam_Score
FROM raw_students
WHERE Attendance >= 90
ORDER BY Exam_Score DESC, student_id
LIMIT 10;
"""
print("4.1 - Top 10 sinh viên chuyên cần >= 90%:\n", pd.read_sql_query(sql_4, conn).to_string(index=False))

sql_4b = """
SELECT School_Type, Gender, COUNT(*) AS students,
       ROUND(AVG(Exam_Score), 2) AS avg_exam_score
FROM raw_students
GROUP BY School_Type, Gender
ORDER BY School_Type, Gender;
"""
res_4b = pd.read_sql_query(sql_4b, conn)
print("\n4.2 - SQL: số sinh viên theo loại trường & giới tính:\n", res_4b.to_string(index=False))
py_4b = (raw.groupby(["School_Type", "Gender"])["Exam_Score"]
            .agg(students="size", avg_exam_score="mean").round(2).reset_index())
assert (res_4b["students"].values == py_4b["students"].values).all()
print("Đối chiếu Pandas groupby: khớp số lượng từng nhóm")

4.1 - Top 10 sinh viên chuyên cần >= 90%:
  student_id Gender  Hours_Studied  Attendance Motivation_Level Parental_Education_Level  Exam_Score
       1526 Female             27          98              Low              High School         101
       3580 Female             14          90           Medium              High School          99
       4193 Female             28          90           Medium                  College          98
       6348   Male             28          96             High              High School          98
       5967   Male             25          99           Medium              High School          97
       3458 Female             18          93              Low              High School          96
       6523 Female             18          90              Low              High School          95
        771 Female             24          96           Medium              High School          94
       3125 Female             19          90            

## Bước 5 — Data Understanding: Kiểm tra chất lượng dữ liệu bằng SQL & Python
Kiểm tra số dòng, bản ghi trùng toàn bộ, giá trị NULL theo cột, phạm vi giá trị hợp lý (điểm/chuyên cần 0–100, giờ ngủ 0–24, giờ học/tuần 0–168) và các giá trị của biến phân loại.

In [4]:
checks = {
 "Tổng số dòng": "SELECT COUNT(*) AS n FROM raw_students",
 "Bản ghi trùng toàn bộ (SQL)": """
    SELECT COUNT(*) AS duplicate_rows FROM (
      SELECT COUNT(*) AS c FROM raw_students
      GROUP BY Hours_Studied, Attendance, Parental_Involvement, Access_to_Resources,
               Extracurricular_Activities, Sleep_Hours, Previous_Scores, Motivation_Level,
               Family_Income, Teacher_Quality, School_Type, Peer_Influence,
               Physical_Activity, Parental_Education_Level, Gender, Exam_Score
      HAVING COUNT(*) > 1)""",
 "NULL / rỗng theo cột phân loại": """
    SELECT SUM(Teacher_Quality IS NULL OR TRIM(Teacher_Quality) = '') AS null_teacher_quality,
           SUM(Parental_Education_Level IS NULL OR TRIM(Parental_Education_Level) = '') AS null_parent_edu,
           SUM(Parental_Involvement IS NULL) AS null_parent_involvement,
           SUM(Motivation_Level IS NULL) AS null_motivation
    FROM raw_students""",
 "Giá trị ngoài phạm vi": """
    SELECT SUM(Exam_Score NOT BETWEEN 0 AND 100)      AS exam_out_0_100,
           SUM(Previous_Scores NOT BETWEEN 0 AND 100) AS prev_out_0_100,
           SUM(Attendance NOT BETWEEN 0 AND 100)      AS attendance_out_0_100,
           SUM(Sleep_Hours NOT BETWEEN 0 AND 24)      AS sleep_out_0_24,
           SUM(Hours_Studied NOT BETWEEN 0 AND 168)   AS hours_out_0_168
    FROM raw_students""",
 "Bản ghi có điểm thi > 100": "SELECT rowid AS student_id, Hours_Studied, Attendance, Previous_Scores, Exam_Score FROM raw_students WHERE Exam_Score > 100",
 "Min/Max các biến số": """
    SELECT MIN(Hours_Studied) min_hours, MAX(Hours_Studied) max_hours,
           MIN(Attendance) min_att, MAX(Attendance) max_att,
           MIN(Exam_Score) min_exam, MAX(Exam_Score) max_exam
    FROM raw_students""",
 "Giá trị của Parental_Education_Level": """
    SELECT COALESCE(Parental_Education_Level, '<NULL>') AS value, COUNT(*) AS n
    FROM raw_students GROUP BY Parental_Education_Level ORDER BY n DESC"""
}
for label, query in checks.items():
    print("\n", label, "\n", pd.read_sql_query(query, conn).to_string(index=False))
print("\nNULL theo cột (Python):\n", raw.isna().sum()[raw.isna().sum() > 0].to_string())
print("Bản ghi trùng toàn bộ (Python):", raw.duplicated().sum())
print("Điểm thi ngoài 0-100 (Python):", (~raw["Exam_Score"].between(0, 100)).sum())


 Tổng số dòng 
    n
6607

 Bản ghi trùng toàn bộ (SQL) 
  duplicate_rows
              0

 NULL / rỗng theo cột phân loại 
  null_teacher_quality  null_parent_edu  null_parent_involvement  null_motivation
                   78               90                        0                0

 Giá trị ngoài phạm vi 
  exam_out_0_100  prev_out_0_100  attendance_out_0_100  sleep_out_0_24  hours_out_0_168
              1               0                     0               0                0

 Bản ghi có điểm thi > 100 
  student_id  Hours_Studied  Attendance  Previous_Scores  Exam_Score
       1526             27          98               93         101

 Min/Max các biến số 
  min_hours  max_hours  min_att  max_att  min_exam  max_exam
         1         44       60      100        55       101

 Giá trị của Parental_Education_Level 
        value    n
 High School 3223
     College 1989
Postgraduate 1305
      <NULL>   90

NULL theo cột (Python):
 Teacher_Quality             78
Parental_Educa

## Bước 6 — Data Preparation: Làm sạch bằng SQL + Python (PHẦN TRỌNG TÂM)
**Nguyên tắc:**
1. Tạo khóa `student_id` = thứ tự dòng (rowid).
2. Chuẩn hóa tên cột về dạng chữ thường (`snake_case`) và `TRIM` văn bản.
3. **Không tự ý thay đổi điểm thi**; giá trị ngoài phạm vi hợp lệ (ví dụ `Exam_Score = 101`) được đánh dấu thành `NULL` để kiểm tra.
4. Điền `Unknown` cho `Teacher_Quality` và `Parental_Education_Level` bị thiếu (không đoán giá trị).
5. Loại bản ghi trùng toàn bộ nếu có (giữ bản ghi đầu tiên).

Câu lệnh SQL dưới đây thực hiện làm sạch trên **bảng mới** `students_clean_sql`, không sửa dữ liệu gốc. Python thực hiện cùng quy tắc để đối chiếu.

In [5]:
conn.executescript("""
DROP TABLE IF EXISTS students_clean_sql;
CREATE TABLE students_clean_sql AS
WITH standardized AS (
 SELECT rowid AS student_id,
        TRIM(Gender) AS gender,
        CASE WHEN Hours_Studied BETWEEN 0 AND 168 THEN Hours_Studied ELSE NULL END AS hours_studied,
        CASE WHEN Attendance BETWEEN 0 AND 100 THEN Attendance ELSE NULL END AS attendance,
        CASE WHEN Sleep_Hours BETWEEN 0 AND 24 THEN Sleep_Hours ELSE NULL END AS sleep_hours,
        CASE WHEN Previous_Scores BETWEEN 0 AND 100 THEN Previous_Scores ELSE NULL END AS previous_scores,
        Physical_Activity AS physical_activity,
        TRIM(Parental_Involvement) AS parental_involvement,
        TRIM(Access_to_Resources) AS access_to_resources,
        TRIM(Extracurricular_Activities) AS extracurricular_activities,
        TRIM(Motivation_Level) AS motivation_level,
        TRIM(Family_Income) AS family_income,
        COALESCE(NULLIF(TRIM(Teacher_Quality), ''), 'Unknown') AS teacher_quality,
        TRIM(School_Type) AS school_type,
        TRIM(Peer_Influence) AS peer_influence,
        COALESCE(NULLIF(TRIM(Parental_Education_Level), ''), 'Unknown') AS parental_education_level,
        CASE WHEN Exam_Score BETWEEN 0 AND 100 THEN Exam_Score ELSE NULL END AS exam_score,
        ROW_NUMBER() OVER (
            PARTITION BY Hours_Studied, Attendance, Parental_Involvement, Access_to_Resources,
                         Extracurricular_Activities, Sleep_Hours, Previous_Scores, Motivation_Level,
                         Family_Income, Teacher_Quality, School_Type, Peer_Influence,
                         Physical_Activity, Parental_Education_Level, Gender, Exam_Score
            ORDER BY rowid) AS rn
 FROM raw_students
)
SELECT student_id, gender, hours_studied, attendance, sleep_hours, previous_scores,
       physical_activity, parental_involvement, access_to_resources,
       extracurricular_activities, motivation_level, family_income, teacher_quality,
       school_type, peer_influence, parental_education_level, exam_score
FROM standardized WHERE rn = 1
ORDER BY student_id;
""")
clean_sql = pd.read_sql_query("SELECT * FROM students_clean_sql ORDER BY student_id", conn)
print("SQL - số dòng sạch:", len(clean_sql))
print("SQL - Teacher_Quality Unknown:", (clean_sql.teacher_quality == "Unknown").sum())
print("SQL - Parental_Education Unknown:", (clean_sql.parental_education_level == "Unknown").sum())
print("SQL - exam_score bị đánh dấu NULL:", clean_sql.exam_score.isna().sum())

SQL - số dòng sạch: 6607
SQL - Teacher_Quality Unknown: 78
SQL - Parental_Education Unknown: 90
SQL - exam_score bị đánh dấu NULL: 1


In [6]:
COLS = ["student_id", "gender", "hours_studied", "attendance", "sleep_hours", "previous_scores",
        "physical_activity", "parental_involvement", "access_to_resources",
        "extracurricular_activities", "motivation_level", "family_income", "teacher_quality",
        "school_type", "peer_influence", "parental_education_level", "exam_score"]

clean_py = raw.copy()
clean_py.insert(0, "student_id", np.arange(1, len(clean_py) + 1))   # = rowid trong SQLite
clean_py = clean_py[~clean_py.drop(columns="student_id").duplicated(keep="first")]
clean_py.columns = clean_py.columns.str.strip().str.lower()
for column in clean_py.select_dtypes(include=["object", "string"]).columns:
    clean_py[column] = clean_py[column].astype("string").str.strip().replace("", pd.NA)
for column in ["teacher_quality", "parental_education_level"]:
    clean_py[column] = clean_py[column].fillna("Unknown")
for column, low, high in [("hours_studied", 0, 168), ("sleep_hours", 0, 24),
                          ("attendance", 0, 100), ("previous_scores", 0, 100),
                          ("exam_score", 0, 100)]:
    clean_py[column] = pd.to_numeric(clean_py[column], errors="coerce").astype("Int64")
    clean_py.loc[~clean_py[column].between(low, high), column] = np.nan
clean_py = clean_py[COLS].sort_values("student_id").reset_index(drop=True)
print("Python - số dòng sạch:", len(clean_py))
print("Python - số ô thiếu còn lại:\n", clean_py.isna().sum()[clean_py.isna().sum() > 0].to_string())
print("SQL vs Python - Teacher_Quality Unknown:",
      (clean_sql.teacher_quality == "Unknown").sum(), (clean_py.teacher_quality == "Unknown").sum())
print("SQL vs Python - Parental_Education Unknown:",
      (clean_sql.parental_education_level == "Unknown").sum(), (clean_py.parental_education_level == "Unknown").sum())
_ = clean_py.to_sql("students_clean_python", conn, if_exists="replace", index=False)

Python - số dòng sạch: 6607
Python - số ô thiếu còn lại:
 exam_score    1
SQL vs Python - Teacher_Quality Unknown: 78 78
SQL vs Python - Parental_Education Unknown: 90 90


## Bước 7 — Modeling: Phân nhóm quy tắc bằng SQL và Python
Để ưu tiên SQL/Python, bước “Modeling” sử dụng **mô hình phân nhóm quy tắc** (`CASE WHEN`) theo số giờ tự học/tuần (`≤10h`, `11–20h`, `21–30h`, `>30h`), không cần học máy. Window function `RANK()` xếp hạng nhóm và `AVG() OVER ()` tính chênh lệch so với trung bình toàn bộ. Python dùng `np.select` cho cùng quy tắc và tính tương quan Pearson.

In [7]:
sql_7 = """
WITH grouped AS (
 SELECT CASE WHEN hours_studied <= 10 THEN '1. <=10h'
             WHEN hours_studied <= 20 THEN '2. 11-20h'
             WHEN hours_studied <= 30 THEN '3. 21-30h'
             ELSE '4. >30h' END AS study_group,
        exam_score, attendance
 FROM students_clean_sql
 WHERE hours_studied IS NOT NULL AND exam_score IS NOT NULL
), summary AS (
 SELECT study_group, COUNT(*) AS student_count,
        ROUND(AVG(exam_score), 2) AS avg_exam_score,
        ROUND(AVG(attendance), 2) AS avg_attendance
 FROM grouped GROUP BY study_group
)
SELECT study_group, student_count, avg_exam_score, avg_attendance,
       ROUND(avg_exam_score - SUM(avg_exam_score * student_count) OVER () / SUM(student_count) OVER (), 2) AS diff_vs_overall,
       RANK() OVER (ORDER BY avg_exam_score DESC) AS score_rank
FROM summary
ORDER BY study_group;
"""
result_study = pd.read_sql_query(sql_7, conn)
print("SQL - Điểm theo nhóm giờ tự học/tuần:\n", result_study.to_string(index=False))

clean_py["study_group"] = np.select(
    [clean_py.hours_studied <= 10, clean_py.hours_studied <= 20, clean_py.hours_studied <= 30],
    ["1. <=10h", "2. 11-20h", "3. 21-30h"], default="4. >30h")
py_study = (clean_py.dropna(subset=["exam_score"]).groupby("study_group")["exam_score"]
            .agg(student_count="size", avg_exam_score="mean").round(2).reset_index())
print("\nPython - cùng quy tắc phân nhóm:\n", py_study.to_string(index=False))

sql_7b = """
SELECT motivation_level,
       ROUND(AVG(CASE WHEN parental_involvement = 'Low'    THEN exam_score END), 2) AS parent_low,
       ROUND(AVG(CASE WHEN parental_involvement = 'Medium' THEN exam_score END), 2) AS parent_medium,
       ROUND(AVG(CASE WHEN parental_involvement = 'High'   THEN exam_score END), 2) AS parent_high
FROM students_clean_sql
GROUP BY motivation_level
ORDER BY CASE motivation_level WHEN 'Low' THEN 1 WHEN 'Medium' THEN 2 ELSE 3 END;
"""
print("\nSQL - Điểm TB theo Động lực (dòng) × Sự tham gia của phụ huynh (cột):\n",
      pd.read_sql_query(sql_7b, conn).to_string(index=False))

num_cols = ["hours_studied", "attendance", "sleep_hours", "previous_scores", "physical_activity", "exam_score"]
print("\nPython - Tương quan Pearson với exam_score:\n",
      clean_py[num_cols].corr()["exam_score"].drop("exam_score").sort_values(ascending=False).round(3).to_string())

SQL - Điểm theo nhóm giờ tự học/tuần:
 study_group  student_count  avg_exam_score  avg_attendance  diff_vs_overall  score_rank
   1. <=10h            365           63.97           80.15            -3.26           4
  2. 11-20h           3179           66.18           80.05            -1.05           3
  3. 21-30h           2809           68.51           79.94             1.28           2
    4. >30h            253           71.01           79.13             3.78           1

Python - cùng quy tắc phân nhóm:
 study_group  student_count  avg_exam_score
   1. <=10h            365           63.97
  2. 11-20h           3179           66.18
  3. 21-30h           2809           68.51
    4. >30h            253           71.01

SQL - Điểm TB theo Động lực (dòng) × Sự tham gia của phụ huynh (cột):
 motivation_level  parent_low  parent_medium  parent_high
             Low       65.81          66.51        67.71
          Medium       66.41          67.20        68.21
            High       66.82

## Bước 8 — Evaluation: Đánh giá và đối chiếu bằng SQL/Python
Đối chiếu số dòng, điểm trung bình và kết quả phân nhóm giữa SQL và Python; đánh giá dữ liệu còn thiếu. Dùng `JOIN` giữa bảng sạch và CTE trung bình theo nhóm để so sánh từng nhóm chuyên cần, học vấn phụ huynh và ngoại khóa với trung bình chung. Không kết luận yếu tố nào “gây ra” điểm số.

In [8]:
sql_8 = """
SELECT COUNT(*) AS total_students, ROUND(AVG(exam_score), 3) AS avg_score,
       SUM(CASE WHEN parental_education_level = 'Unknown' THEN 1 ELSE 0 END) AS unknown_parent_edu,
       SUM(CASE WHEN teacher_quality = 'Unknown' THEN 1 ELSE 0 END) AS unknown_teacher_quality,
       SUM(CASE WHEN exam_score IS NULL THEN 1 ELSE 0 END) AS missing_exam_score
FROM students_clean_sql;
"""
summary_sql = pd.read_sql_query(sql_8, conn)
print("SQL kiểm tra:\n", summary_sql.to_string(index=False))
print("Python điểm trung bình:", round(clean_py.exam_score.mean(), 3))
assert len(clean_sql) == len(clean_py), "SQL và Python không đồng nhất số bản ghi"
assert abs(clean_sql.exam_score.mean() - clean_py.exam_score.mean()) < 1e-8
assert (clean_sql.student_id.values == clean_py.student_id.values).all()
assert (result_study.avg_exam_score.values == py_study.avg_exam_score.values).all()
print("Đối chiếu SQL/Python (số dòng, điểm TB, student_id, phân nhóm giờ học): ĐẠT")

print("\nSQL - Điểm theo tỷ lệ chuyên cần:\n", pd.read_sql_query("""
SELECT CASE WHEN attendance < 70 THEN '1. Dưới 70%'
            WHEN attendance < 80 THEN '2. 70-<80%'
            WHEN attendance < 90 THEN '3. 80-<90%' ELSE '4. 90-100%' END AS attendance_group,
       COUNT(*) AS students, ROUND(AVG(exam_score), 2) AS avg_score
FROM students_clean_sql GROUP BY attendance_group ORDER BY attendance_group
""", conn).to_string(index=False))

print("\nSQL (CTE + JOIN) - Học vấn phụ huynh × Ngoại khóa so với trung bình chung:\n", pd.read_sql_query("""
WITH grp AS (
  SELECT parental_education_level, extracurricular_activities,
         COUNT(*) AS students, AVG(exam_score) AS avg_score
  FROM students_clean_sql
  GROUP BY parental_education_level, extracurricular_activities
), overall AS (
  SELECT AVG(exam_score) AS overall_avg FROM students_clean_sql
)
SELECT g.parental_education_level, g.extracurricular_activities, g.students,
       ROUND(g.avg_score, 2) AS avg_score,
       ROUND(g.avg_score - o.overall_avg, 2) AS diff_vs_overall
FROM grp g CROSS JOIN overall o
ORDER BY avg_score DESC
""", conn).to_string(index=False))

print("\nSQL (JOIN bảng sạch với trung bình nhóm động lực) - 5 sinh viên vượt trung bình nhóm nhiều nhất:\n",
      pd.read_sql_query("""
WITH motiv AS (
  SELECT motivation_level, AVG(exam_score) AS group_avg
  FROM students_clean_sql GROUP BY motivation_level
)
SELECT s.student_id, s.motivation_level, s.hours_studied, s.attendance, s.exam_score,
       ROUND(m.group_avg, 2) AS group_avg,
       ROUND(s.exam_score - m.group_avg, 2) AS above_group_avg
FROM students_clean_sql s
JOIN motiv m ON s.motivation_level = m.motivation_level
WHERE s.exam_score IS NOT NULL
ORDER BY above_group_avg DESC, s.student_id
LIMIT 5
""", conn).to_string(index=False))

SQL kiểm tra:
  total_students  avg_score  unknown_parent_edu  unknown_teacher_quality  missing_exam_score
           6607     67.231                  90                       78                   1
Python điểm trung bình: 67.231
Đối chiếu SQL/Python (số dòng, điểm TB, student_id, phân nhóm giờ học): ĐẠT

SQL - Điểm theo tỷ lệ chuyên cần:
 attendance_group  students  avg_score
     1. Dưới 70%      1573      64.21
      2. 70-<80%      1681      66.28
      3. 80-<90%      1607      68.04
      4. 90-100%      1746      70.12

SQL (CTE + JOIN) - Học vấn phụ huynh × Ngoại khóa so với trung bình chung:
 parental_education_level extracurricular_activities  students  avg_score  diff_vs_overall
            Postgraduate                        Yes       774      68.15             0.92
            Postgraduate                         No       531      67.71             0.48
                 College                        Yes      1208      67.48             0.25
                 Unknown       

## Bước 9 — Deployment: Xuất kết quả để báo cáo
Lưu dữ liệu sạch và bảng phân tích ra CSV; database SQLite `student_analysis.db` chứa `raw_students`, `students_clean_sql`, `students_clean_python`. File `TruyVan_LamSach_PhanTich.sql` chứa các câu lệnh SQLite chạy độc lập.

In [9]:
from datetime import datetime

def save_csv(df, filename):
    """Ghi CSV; nếu file đang bị khóa (mở trong Excel / OneDrive đang đồng bộ) thì ghi sang tên khác thay vì dừng lỗi."""
    try:
        df.to_csv(filename, index=False, encoding="utf-8-sig")
        print(f"  ✔ {filename}")
    except PermissionError:
        alt = filename.replace(".csv", f"_{datetime.now():%Y%m%d_%H%M%S}.csv")
        df.to_csv(alt, index=False, encoding="utf-8-sig")
        print(f"  ⚠ {filename} đang được mở ở chương trình khác (Excel?) -> đã lưu sang {alt}. Hãy đóng file rồi chạy lại ô này.")

save_csv(clean_py.drop(columns="study_group", errors="ignore"), "student_performance_cleaned_python.csv")
save_csv(clean_sql, "student_performance_cleaned_sql.csv")
save_csv(result_study, "analysis_study_groups.csv")
conn.commit()
print("Xuất 3 file CSV và student_analysis.db thành công")

  ✔ student_performance_cleaned_python.csv
  ✔ student_performance_cleaned_sql.csv
  ✔ analysis_study_groups.csv
Xuất 3 file CSV và student_analysis.db thành công


## Bước 10 — Feedback: Kết luận, kiểm tra lại và cải tiến
- Dữ liệu khá sạch: không có bản ghi trùng; 78 ô thiếu `Teacher_Quality` và 90 ô thiếu `Parental_Education_Level` được gắn `Unknown`; 1 bản ghi `Exam_Score = 101` (ngoài 0–100) được đánh dấu `NULL`, không bị sửa tùy ý.
- Số giờ tự học/tuần, tỷ lệ chuyên cần, mức động lực và sự tham gia của phụ huynh có liên hệ thuận với điểm thi; chuyên cần là yếu tố có tương quan mạnh nhất.
- Tiếp tục kiểm tra dữ liệu thiếu/ngoại lệ khi nạp phiên bản mới; cần hỏi lại nguồn dữ liệu về bản ghi điểm 101.
- Có thể bổ sung truy vấn theo thu nhập gia đình, chất lượng giáo viên, ảnh hưởng bạn bè. Dữ liệu không có khóa gốc nên `student_id` chỉ có ý nghĩa theo thứ tự dòng — không ghép với bộ dữ liệu khác theo vị trí dòng.

In [10]:
print("KẾT QUẢ CUỐI CÙNG")
print("Số bản ghi:", len(clean_py))
print("Điểm thi trung bình:", round(clean_py.exam_score.mean(), 2))
print("Giờ tự học/tuần trung bình:", round(clean_py.hours_studied.mean(), 2))
print("Chuyên cần trung bình (%):", round(clean_py.attendance.mean(), 2))
print("Điểm theo nhóm giờ học:\n", result_study.to_string(index=False))
conn.close()

KẾT QUẢ CUỐI CÙNG
Số bản ghi: 6607
Điểm thi trung bình: 67.23
Giờ tự học/tuần trung bình: 19.98
Chuyên cần trung bình (%): 79.98
Điểm theo nhóm giờ học:
 study_group  student_count  avg_exam_score  avg_attendance  diff_vs_overall  score_rank
   1. <=10h            365           63.97           80.15            -3.26           4
  2. 11-20h           3179           66.18           80.05            -1.05           3
  3. 21-30h           2809           68.51           79.94             1.28           2
    4. >30h            253           71.01           79.13             3.78           1
